# run_e39_e31
Phase 3 Stage B3 Worker 1 (`E39+E31`): CPU scoring & mechanistic analysis for sub-caption token positions (`E39`), decoder visual-token semantic probes, and attention-mass gap correlations (`E31`).

- Configure only Cell 3 (`MODEL = 'qwen'` or `'gemma'`, `PLAN_REF`, optional `RELEASE_REF` / `RESUME_ANALYSIS_REF`).
- Uses the pinned Python 3.11/3.12 CPU runtime (`requirements-stage2-cpu.txt`) with single-threaded BLAS/OpenMP.
- Authenticates `phase3_gpu_plan` (`bind=True`, `e35_excluded=True`, `confirm_excluded=True`) and Stage B2 `E39+E31` release (`verify_e39_e31_release`), enforces Stage 2 draw-0 feature and `en->en` matched logit parity (`<= 1e-4`), scores all 8 `E39_SITES` through both frozen Stage 3 head types (`last_caption`, `caption_mean`) across `4 depths x 2 conditions x 16 routes`, runs the 5-fold CV `decoder_visual_mean` semantic probe (`encoder -> projector -> decoder:d1..d4`) and attention-gap correlation analysis, and publishes under `phase3/results/e39_e31/<model>/`. Phase 1 & Phase 2 remain strictly read-only and `CONFIRM` stays closed.


In [ ]:
import os, sys, subprocess
from pathlib import Path

if sys.version_info[:2] not in ((3, 11), (3, 12)):
    raise RuntimeError('Use Python 3.11/3.12 with the pinned CPU requirements.')
for name in ('OPENBLAS_NUM_THREADS', 'OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[name] = '1'

from google.colab import drive
drive.mount('/content/drive')

REPO = Path('/content/Research-phase3-e39-e31')
CODE_REF = 'phase3'
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Drazr/Research.git', str(REPO)], check=True)
if subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip():
    raise RuntimeError('Use a clean checkout; preserve local work before continuing.')
subprocess.run(['git', 'fetch', 'origin', CODE_REF], cwd=REPO, check=True)
CODE_COMMIT = subprocess.check_output(['git', 'rev-parse', 'FETCH_HEAD'], cwd=REPO, text=True).strip()
subprocess.run(['git', 'checkout', '--detach', CODE_COMMIT], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'requirements-stage2-cpu.txt')], check=True)
print('Fixed code commit:', CODE_COMMIT)
print('Restart if installation requests it, then rerun setup before continuing.')


In [ ]:
# Visible Stage B3 E39+E31 settings (edit only this cell).
PHASE1_ROOT = Path('/content/drive/MyDrive/phase1')
MODEL = 'qwen'              # 'qwen' or 'gemma' (run one runtime per model).
PLAN_REF = {
    'path': 'phase3/plans/20261120T060305Z-11b534d878f84475a50b561efbd06650/_COMPLETE.json',
    'sha256': '3603f36631ec8165445613e08caf2a3b74bd7626f84f79fdde59fc34f0088b1c',
}
RELEASE_REF = None          # Optional override; None defaults to CERTIFIED_STAGE_B2_RELEASES['e39_e31'][MODEL].
RESUME_ANALYSIS_REF = None  # Optional explicit {'path': ..., 'sha256': ...} or Path to phase3_analysis_block _COMPLETE.json.
OUTPUT_REF = None           # Optional mounted JSON path (e.g. PHASE1_ROOT / f'phase3/results/e39_e31_{MODEL}_ref.json').


In [ ]:
import json, signal

if subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip() != CODE_COMMIT:
    raise RuntimeError('Checkout changed after setup.')
if subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip():
    raise RuntimeError('Use a clean Phase 3 checkout.')
if MODEL not in ('qwen', 'gemma'):
    raise RuntimeError("Set MODEL to 'qwen' or 'gemma'.")
if not PHASE1_ROOT.is_dir():
    raise RuntimeError(f'Missing mounted PHASE1_ROOT directory: {PHASE1_ROOT}')

def _format_ref(ref, label):
    if isinstance(ref, dict):
        target = PHASE1_ROOT / ref['path']
        if not target.is_file():
            raise RuntimeError(f'Missing {label} manifest: {target}')
        return json.dumps(ref)
    target = Path(ref)
    if not target.is_file():
        raise RuntimeError(f'Missing {label} manifest: {target}')
    return str(target)

command = [
    sys.executable, '-u', '-m', 'src.phase3.run_e39_e31',
    '--phase1-root', str(PHASE1_ROOT),
    '--model', MODEL,
    '--storage-scope', 'persistent-drive',
]
if PLAN_REF is not None:
    command += ['--plan', _format_ref(PLAN_REF, 'phase3_gpu_plan')]
if RELEASE_REF is not None:
    command += ['--release', _format_ref(RELEASE_REF, 'e39_e31_release')]
if RESUME_ANALYSIS_REF is not None:
    command += ['--resume-analysis', _format_ref(RESUME_ANALYSIS_REF, 'resume_analysis')]
if OUTPUT_REF is not None:
    command += ['--output-ref', str(OUTPUT_REF)]

process = subprocess.Popen(
    command, cwd=REPO, env=os.environ.copy(),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
blockers = []
events = []
try:
    for line in process.stdout:
        print(line, end='', flush=True)
        try:
            payload = json.loads(line)
            if isinstance(payload, dict):
                events.append(payload)
                if payload.get('status') == 'blocked':
                    blockers.extend(payload.get('blockers', []))
        except json.JSONDecodeError:
            pass
except KeyboardInterrupt:
    process.send_signal(signal.SIGINT)
    for line in process.stdout:
        print(line, end='', flush=True)
    process.wait()
    raise
code = process.wait()
if code != 0:
    detail = '; '.join(blockers) if blockers else f'exit code {code}'
    raise RuntimeError(f'E39+E31 analysis worker blocked/failed for {MODEL}: {detail}')

completed = [e for e in events if e.get('event') == 'e39_e31_complete']
if completed:
    print('\n=== E39+E31 ANALYSIS REFERENCE ===')
    print(json.dumps(completed[-1], indent=2))
